# SiQUID — visualization sandbox (playback mock)

Sandbox for the dashboard figures. **Figure logic lives in `siquid_monitor/` (data.py + figures.py)** —
this notebook only loads data and calls the builders, so the notebook and the Dash app share one source of truth.

- **`docs/figures.md`** — figure-design principles & panel overview.
- **`docs/data.md`** — recorded-data overview.
- **`docs/monitoring.md`** — monitoring goals (D5.3).
- Full working docs (data schema, caveats, roadmap) live in `internal/` (not published).
- **D4.4 deliverable** (`resources/D4.4_*.docx`) — reference; its **Figure 13** is the de-facto
  layout (these panels + a CHSH **S-value** panel this dataset lacks).

Honest-replay reminders (enforced in `figures.py`): the recorded link **never entangled**; counts are
**delay-biased** and **not accidental-subtracted**; grey shading = optimizer actively searching.


## 1. Setup — import the shared module

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from siquid_monitor import figures as F
from siquid_monitor.data import CORRELATED, ERRORS, LABELS, channel_singles, load_repo_dataset

print("imported siquid_monitor; GAP_S =", F.GAP_S, "median window =", F.MEDIAN_W)

In [ ]:
ds = load_repo_dataset()  # DEFAULT_DATA_DIR = ../external/long-distance-entanglement/Data
m, v = ds.measurements, ds.voltages
print(ds.name)
print(f"measurements: {len(m)} rows  {m.t.min()} .. {m.t.max()}")
print(f"voltages:     {len(v)} rows")
m[
    [
        "t",
        "visibility",
        "vis_HV",
        "vis_DA",
        "QBER_total",
        "total_coincidences",
        "coinc_rate",
        "has_voltage",
    ]
].head()

## 2. Interconnection / coverage recap
Shared time axis (no row-merge). `has_voltage` marks measurements paired with an optimizer voltage row (≤1 s).

In [ ]:
print(
    "rows with voltage (<=1s):",
    int(m.has_voltage.sum()),
    "/",
    len(m),
    f"({100 * m.has_voltage.mean():.1f}%)",
)
gaps = np.diff(m.timestamp.values)
print(f"median cadence {np.median(gaps):.1f}s; genuine gaps >{F.GAP_S:g}s: {(gaps > F.GAP_S).sum()}")
print("optimizer-active spans (merged):", len(F.optimizer_spans(m)))

## 3. Candidate panels (built by `siquid_monitor.figures`)

Four gate-2 bundles. **Headline** is the PoC target; the rest are directions. Edit `figures.py` to change any.

### 3a. Headline QKD — visibility + QBER  (PoC target)

In [ ]:
F.fig_headline(m)

### 3b. Source / link health

In [ ]:
F.fig_source(m)

### 3c. Stability & drift

In [ ]:
F.fig_stability(m, v)

### 3d. Diagnostics

In [ ]:
F.fig_diagnostics(m)

## 4. Playback experiment

Virtual clock exposes only samples up to "virtual now"; speed multiplier maps real→recorded seconds.
Static snapshots here; the Dash app will drive the same filtering with a `dcc.Interval` timer + speed/play-pause.

In [ ]:
class Player:
    def __init__(self, df, speed=2000.0):
        self.df = df.sort_values("timestamp").reset_index(drop=True)
        self.t0 = float(self.df.timestamp.min())
        self.t1 = float(self.df.timestamp.max())
        self.speed = speed
        self.elapsed = 0.0

    def tick(self, real_dt):
        self.elapsed = min(self.elapsed + real_dt * self.speed, self.t1 - self.t0)

    @property
    def now_ts(self):
        return self.t0 + self.elapsed

    def visible(self):
        return self.df[self.df.timestamp <= self.now_ts]


p = Player(m, speed=2000.0)
total = p.t1 - p.t0
print(f"recorded span {total / 3600:.1f} h; at 2000x -> {total / 2000 / 60:.1f} min real-time replay")
for frac in (0.25, 0.5, 1.0):
    p.elapsed = total * frac
    vis = p.visible()
    last = vis.iloc[-1]
    print(
        f"  virtual t={pd.to_datetime(p.now_ts, unit='s')}  rows={len(vis):5d}  "
        f"vis={last.visibility:+.3f}  QBER={last.QBER_total:.3f}  coinc_rate={last.coinc_rate:.1f}"
    )
p.elapsed = total * 0.5
F.fig_headline(p.visible())

## 5. Observations / directions (fill in while exploring)

- [ ] `GAP_S` (120 s), median window (51), playback speed (2000×) — tune in `figures.py` / here.
- [ ] Visibility y-range −1..1 vs auto; QBER panel kept for audience though `vis = 1 − 2·QBER`.
- [ ] Poisson uncertainty band; accidental-subtracted view.
- [ ] KPI tiles for the Dash app: latest visibility, QBER, coinc rate, total singles, virtual clock.
